*0.2 Math / ML basics*

# cosine similarity

**The situation.** A team builds keyword search with TF-IDF vectors (word counts, weighted). Ranking looks wrong: a 3,000-word policy document comes first for every query, even "reset password". The document simply contains every word many times, so its vector is long, and long vectors win every dot product.

**Cosine similarity.** Divide the dot product by the lengths of both vectors. What remains is only the *angle* between them: 1 means the same direction, 0 means unrelated. Length — how long the text was — drops out. The policy document no longer wins for being long.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Reproduce the bug with real TF-IDF vectors.** `norm=None` keeps the raw lengths, which is where the bug lives. Rank with dot product, then with cosine.

In [2]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Reset your password from the login page.",
    "Refund policy. " * 40
    + "Password and login rules are in section 4. " * 10
    + "Contact support for anything else. " * 40,
    "Change the email on your account.",
]
labels = ["short password article", "long policy document", "email article"]
query = "reset password"

vectorizer = TfidfVectorizer(norm=None)
document_vectors = vectorizer.fit_transform(documents)
query_vector = vectorizer.transform([query])

dot_scores = np.asarray(document_vectors @ query_vector.T.toarray()).ravel()
cosine_scores = cosine_similarity(document_vectors, query_vector).ravel()
lengths = np.asarray(np.sqrt(document_vectors.multiply(document_vectors).sum(axis=1))).ravel()

print(f"{'document':<26}{'length':>8}{'dot':>8}{'cosine':>8}")
for label, length, dot, cos in zip(labels, lengths, dot_scores, cosine_scores):
    print(f"{label:<26}{length:>8.1f}{dot:>8.2f}{cos:>8.2f}")
assert np.argmax(dot_scores) == 1 and np.argmax(cosine_scores) == 0

document                    length     dot  cosine
short password article         3.9    4.52    0.55
long policy document         184.0   16.58    0.04
email article                  3.8    0.00    0.00


**Reading the output.** By dot product the long policy document wins — look at its length column. By cosine the short password article wins, which is the right answer. Same vectors, same query; only the scoring changed.

```
dot product   ──  rewards direction AND length   ──▶  long documents win
cosine        ──  direction only (angle)          ──▶  meaning wins
```

**The rule to remember.** If your vectors are not all the same length, use cosine. If they are (unit-length embeddings), cosine and dot product are the same number, and dot product is cheaper.

| Use it when | Don't when | Instead use |
|---|---|---|
| TF-IDF, bag-of-words, any vectors of varying length | vectors are already unit length — you would divide by 1 | dot product |

**Watch out**
- Cosine of 0.9 is not "90% similar"; it is only useful to rank, not to explain.
- Two vectors that are all zeros (a query with only unknown words) have no angle; expect a 0 or a NaN and handle it.
- Normalising once at index time and using dot product at query time gives cosine results at dot-product speed.